# Supply Chain Data Palm Oil: ETL Pipeline with Airflow Orchestration 
# Pra-Automation

Author      : Ryan Farid Pratama  
Tech Stack  : Python, Pandas, Great Expectation, PySpark

**Project Overview**  
This project aims to build and demonstrate an automated ETL data pipeline system with key objectives are:
1. Data processing to handle data pipeline task ETL (Extract, Transform and Load) using PySpark
2. Perform data validation with Great Expectations
3. Set up DAGs to automate and schedule the execution of data pipeline task using Apache Airflow
4. Data preparing for NoSQL Database with MongoDB

**Architecture Pipeline**  
**A. Pra-Automation**  
1. Setup Environment
2. Data Extraction      : Pandas is used to extract dataset
    - Data Sampling        : Kalimantan Only
    - Data Exploration     : EDA is performed using Pandas to define data cleaning and processing logic
3. Data Transformation  : Drop unused column, drop any duplicate rows if there any
4. Data Validation      : Validate data using Great Expectations

**B. Automation**  
1. Set up DAGs script   : Extract, transform and load
2. Running DAGs script  : Running the scripts in Apache Airflow
3. Data Loading         : Loading the data into Database which MongoDB 

# A. Environment & Setup
Libraries import material in order to running a scripts

In [1]:
# Data Manipulation & DataFrame
import pandas as pd
import numpy as np

# Ignore warning to get clean output
import warnings
warnings.filterwarnings("ignore")

# Install the library Great Expectation Package
# !pip install -q "great-expectations==0.18.19"         
# !pip install great_expectations

from dotenv import load_dotenv
load_dotenv()

# Libraries import & environment setup for pySpark
import os, sys

from pyspark.sql import SparkSession                    # PySpark module import for create session
from pyspark.sql import functions as F                  # PySpark module import for functions/columns manipulation
from pyspark.sql.window import Window                   # PySpark module import for windowing calculations
from pyspark.sql.functions import col, count, when      # PySpark module import for counting total NULL of each column

# Create SparkSession
spark = SparkSession\
    .builder\
    .appName("Python Spark PostgreSQL")\
    .config("spark.jars", "path to jbdc driver .jar file")\
    .getOrCreate()

# B. Data Extraction
## B.1 Raw Data Loading
In this sub-stage, dataset is imported using Pandas into Pandas DataFrame <br>

URL             : https://trase.earth/open-data/datasets/supply-chains-indonesia-palm-oil <br>
Dataset Name    : Indonesia Palm Oil - Supply Chain Data <br>
Chandra, A., Benedict, J., Kamaluddin, A., Putri, S. M., Suavet, C., Gollnow, F., Biddle, H., Mueller, C., Ribeiro, V., & Heilmayr, R. (2026). Indonesia palm oil supply chain: Data and Methods (2018–2024) (Version 1.2.5) [Dataset]. Trase. https://doi.org/10.48650/0717-AJ12

In [2]:
# Import and read the .csv file dataset
# 1. Data Loading Raw Palm Oil into PySpark DataFrame, Kalimantan Only
df_palm_oil = spark.read.csv("workspace/data/raw_dataset.csv", header=True, inferSchema=True)\
            .filter(col("province_of_production").contains("KALIMANTAN"))
df_palm_oil.show(100)

+----+------------------+--------------------+---------------------+----------------------+-----------------------+----------------+--------------------+-------------------+--------------------+--------------+--------------------+-------------------+-------------------------------------+--------------+--------------------+--------------------+-----------------------+--------------------+----------------------+-------------------------------+--------------------------------+------------------------------+--------------------------------+---------------------------------------------+--------------------------------------+-----------------------+---------------------------+-------------------------+----------------------------+-------------------------------+-------------------+------------------+--------------------+
|year|     mill_trase_id|   refinery_trase_id|country_of_production|province_of_production|kabupaten_of_production|    product_type|                mill|         mill_group|

## B.2 Data Exploration/EDA (Exploration Data Analysis)
In this sub-stage, sample raw dataset is inspected with EDA process to define data cleaning and processing logic

In [3]:
# 1. File Size Check, but convert to Pandas DataFrame first
df_palm_oil_pd = df_palm_oil.toPandas()
df_palm_oil_pd.info() # Showing summary information of DataFrame structure, such as column data type, non-null count and memory usage

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 588476 entries, 0 to 588475
Data columns (total 34 columns):
 #   Column                                         Non-Null Count   Dtype  
---  ------                                         --------------   -----  
 0   year                                           588476 non-null  int32  
 1   mill_trase_id                                  588476 non-null  object 
 2   refinery_trase_id                              588476 non-null  object 
 3   country_of_production                          588476 non-null  object 
 4   province_of_production                         588476 non-null  object 
 5   kabupaten_of_production                        588476 non-null  object 
 6   product_type                                   588476 non-null  object 
 7   mill                                           588476 non-null  object 
 8   mill_group                                     588476 non-null  object 
 9   refinery                             

File has been reduced by 87,4%, not required enough to commit to github (maximum size is 100mb). Need to explore and transform using PySpark.

In [4]:
# 2. Data Viewing Top 5
print(f"--Palm Oil Top 5--")
df_palm_oil.show(5)

--Palm Oil Top 5--
+----+------------------+--------------------+---------------------+----------------------+-----------------------+--------------+--------------------+-------------------+-----------+--------------+--------------------+-------------------+-------------------------------------+--------------+--------------------+-------------------+-----------------------+--------------------+----------------------+-------------------------------+--------------------------------+------------------------------+--------------------------------+---------------------------------------------+--------------------------------------+-----------------------+---------------------------+-------------------------+----------------------------+-------------------------------+------------------+------------------+------------------+
|year|     mill_trase_id|   refinery_trase_id|country_of_production|province_of_production|kabupaten_of_production|  product_type|                mill|         mill_grou

In [5]:
# 3. Data Viewing Bottom 5
print(f"--Palm Oil Bottom 5--")
df_palm_oil.tail(5)

--Palm Oil Bottom 5--


[Row(year=2024, mill_trase_id='ID-PALM-MILL-X', refinery_trase_id='ID-PALM-REFINERY-X', country_of_production='INDONESIA', province_of_production='KALIMANTAN UTARA', kabupaten_of_production='UNKNOWN', product_type='REFINED PALM OIL', mill='UNKNOWN', mill_group='UNKNOWN', refinery='UNKNOWN', refinery_group='UNKNOWN', exporter='UNKNOWN', exporter_group='UNKNOWN', zero_deforestation_indonesia_palm_oil='UNKNOWN', port_of_export='UNKNOWN', importer='UNKNOWN', importer_group='UNKNOWN', country_of_first_import='UNITED ARAB EMIRATES', economic_bloc_name='(GCC) Gulf Cooperation Council', economic_bloc_trase_id='GCC', province_of_production_trase_id='ID-65', kabupaten_of_production_trase_id='ID-XXXX', country_of_production_trase_id='ID', country_of_first_import_trase_id='AE', palm_oil_deforestation_10_year_total_exposure=0.0, palm_oil_deforestation_annual_exposure=0.024957358929657343, total_emission_exposure=29.123540167967004, gross_emission_luc_exposure=41.54157042676694, net_emission_luc_exp

In [6]:
# 4. Verify Row Counts
print(f"Total Palm Oil Raw Rows: {df_palm_oil.count()}")

Total Palm Oil Raw Rows: 588476


In [7]:
# 5. Ensure all of data type in PySpark align with DDL design before load it into Database
print(f"--Palm Oil Data Type--")
df_palm_oil.printSchema()

--Palm Oil Data Type--
root
 |-- year: integer (nullable = true)
 |-- mill_trase_id: string (nullable = true)
 |-- refinery_trase_id: string (nullable = true)
 |-- country_of_production: string (nullable = true)
 |-- province_of_production: string (nullable = true)
 |-- kabupaten_of_production: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- mill: string (nullable = true)
 |-- mill_group: string (nullable = true)
 |-- refinery: string (nullable = true)
 |-- refinery_group: string (nullable = true)
 |-- exporter: string (nullable = true)
 |-- exporter_group: string (nullable = true)
 |-- zero_deforestation_indonesia_palm_oil: string (nullable = true)
 |-- port_of_export: string (nullable = true)
 |-- importer: string (nullable = true)
 |-- importer_group: string (nullable = true)
 |-- country_of_first_import: string (nullable = true)
 |-- economic_bloc_name: string (nullable = true)
 |-- economic_bloc_trase_id: string (nullable = true)
 |-- province_of_producti

In [8]:
# 6. Count NULL data
print(f"--Palm Oil Total Null Data--")
df_palm_oil.select([count(when(col(c).isNull(), c)).alias(c) 
                     for c in df_palm_oil.columns]).show()


--Palm Oil Total Null Data--
+----+-------------+-----------------+---------------------+----------------------+-----------------------+------------+----+----------+--------+--------------+--------+--------------+-------------------------------------+--------------+--------+--------------+-----------------------+------------------+----------------------+-------------------------------+--------------------------------+------------------------------+--------------------------------+---------------------------------------------+--------------------------------------+-----------------------+---------------------------+-------------------------+----------------------------+-------------------------------+------+---+-----------+
|year|mill_trase_id|refinery_trase_id|country_of_production|province_of_production|kabupaten_of_production|product_type|mill|mill_group|refinery|refinery_group|exporter|exporter_group|zero_deforestation_indonesia_palm_oil|port_of_export|importer|importer_group|countr

There is no null data in both dataset, so there is no need to fill

In [9]:
# 6. Count distinct rows
print(f"--Palm Oil Total Distinct Rows: {df_palm_oil.distinct().count()}")
if df_palm_oil.count() == df_palm_oil.distinct().count():
    print("All rows is unique")

--Palm Oil Total Distinct Rows: 588476
All rows is unique


In [10]:
# 1. Filter only contain "KALIMANTAN" in province
df_kalimantan = df_palm_oil.filter(
    F.col("province_of_production").rlike("KALIMANTAN"))

# 2. Count sum the most common value of each province
df_grouped = (
    df_kalimantan
    .groupBy("province_of_production")
    .agg(F.count("*").alias("jumlah_baris"))
    .orderBy(F.col("jumlah_baris").desc())
)

# 2. Showing the results
df_grouped.show(truncate=False)

+----------------------+------------+
|province_of_production|jumlah_baris|
+----------------------+------------+
|KALIMANTAN TENGAH     |205582      |
|KALIMANTAN BARAT      |161961      |
|KALIMANTAN TIMUR      |142390      |
|KALIMANTAN SELATAN    |56296       |
|KALIMANTAN UTARA      |22247       |
+----------------------+------------+



## B.3 Conclusion Criteria
In this sub-stage, data cleaning conclusions for transform.py is formulated. Conclusion criteria after EDA is performed are:
1. Filter out/drop rows unrelevant ones
2. Ensure and change numerical column data type like VOLUME, FOB and OIL_PALM_HA into FloatType or DoubleType in PySpark
3. Drop unused column based on data dictionaries

# C. Data Transformation
In this sub-stage, data cleaning of raw table is performed using pySpark. Data cleaning in this process including:
1. Whitelist approach, define which columns to keep based on data dictionaries from main dataset (palm oil)
2. Filtering out "UNKNOWN" value

In [11]:
# 1. Select to keep from main dataset
keep_columns = ["year", "mill_trase_id", "product_type", "mill", "mill_group",
                "province_of_production", "province_of_production_trase_id", "kabupaten_of_production", 
                "product_type", "exporter_group", "importer_group", 
                "zero_deforestation_indonesia_palm_oil", "country_of_first_import", 
                "palm_oil_deforestation_annual_exposure", "volume", "total_emission_exposure"]
df_palm_oil_cleaned = df_palm_oil[keep_columns]
df_palm_oil_cleaned.show(10)

print(f"Total Palm Oil Raw Rows: {df_palm_oil_cleaned.count()}") # Verify Row Counts

+----+------------------+--------------+--------------------+-------------------+----------------------+-------------------------------+-----------------------+--------------+-------------------+-------------------+-------------------------------------+-----------------------+--------------------------------------+------------------+-----------------------+
|year|     mill_trase_id|  product_type|                mill|         mill_group|province_of_production|province_of_production_trase_id|kabupaten_of_production|  product_type|     exporter_group|     importer_group|zero_deforestation_indonesia_palm_oil|country_of_first_import|palm_oil_deforestation_annual_exposure|            volume|total_emission_exposure|
+----+------------------+--------------+--------------------+-------------------+----------------------+-------------------------------+-----------------------+--------------+-------------------+-------------------+-------------------------------------+-----------------------+---

In [12]:
# 2. Filter and aggregate by province in PySpark
df_unknown = df_palm_oil_cleaned.filter(
    col("exporter_group").contains("UNKNOWN") &
    col("importer_group").contains("UNKNOWN")
)

df_grouped_2 = (
    df_unknown
    .groupBy("province_of_production")
    .agg(count("*").alias("total_row"))
    .orderBy(col("total_row").desc())
)

df_grouped_2.show(truncate=False)
print(f"Total Unknown Rows: {df_unknown.count()}")

+----------------------+---------+
|province_of_production|total_row|
+----------------------+---------+
|KALIMANTAN TENGAH     |5664     |
|KALIMANTAN BARAT      |4744     |
|KALIMANTAN TIMUR      |3125     |
|KALIMANTAN SELATAN    |2938     |
|KALIMANTAN UTARA      |1571     |
+----------------------+---------+

Total Unknown Rows: 18042


There are still 18042 rows that consist of UNKNOWN AFFILIATIOIN from both exportir or importir group, in order to get more precise data we need to remove it

In [13]:
df_palm_oil_cleaned = df_palm_oil_cleaned.filter(
    ~(col("exporter_group").contains("UNKNOWN") & col("importer_group").contains("UNKNOWN"))
)

# Verify remaining rows (should be 588,476 - 18,042 = 570,434)
print(f"Remaining Rows: {df_palm_oil_cleaned.count()}")

Remaining Rows: 570434


In [14]:
# 3. File Size Check, but convert to Pandas DataFrame first
df_palm_oil_cleaned_pd = df_palm_oil_cleaned.toPandas()
df_palm_oil_cleaned_pd.info() # Showing summary information of DataFrame structure, such as column data type, non-null count and memory usage

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 570434 entries, 0 to 570433
Data columns (total 16 columns):
 #   Column                                  Non-Null Count   Dtype  
---  ------                                  --------------   -----  
 0   year                                    570434 non-null  int32  
 1   mill_trase_id                           570434 non-null  object 
 2   product_type                            570434 non-null  object 
 3   mill                                    570434 non-null  object 
 4   mill_group                              570434 non-null  object 
 5   province_of_production                  570434 non-null  object 
 6   province_of_production_trase_id         570434 non-null  object 
 7   kabupaten_of_production                 570434 non-null  object 
 8   product_type                            570434 non-null  object 
 9   exporter_group                          570434 non-null  object 
 10  importer_group                          5704

File has been reduced by 94%, it is save enough to commit to github (maximum size is 100mb).

In [15]:
# 4. Cleaned data loading to .CSV
df_palm_oil_cleaned_pd.to_csv("dataset_cleaned.csv", index=False)

# D. Data Validation

## D.1 Instantiate Data Context
A Data Context is the primary entry point for a Great Expectations (GX) deployment, and it provides the configurations and methods for all supporting GX components. As the primary entry point for the GX API, the Data Context provides a convenient method for accessing common objects based on untyped input or common defaults. A Data Context also allows us to configure top-level components, and we can use different storage methodologies to back up our Data Context configuration.

In [16]:
# Create a data context using the updated method
import great_expectations as gx
import great_expectations.expectations as gxe
context = gx.get_context(mode='file', project_root_dir='./')

## D.2 Connect to A Datasource
Datasource : provides a standard API for accessing and interacting with data from a wide variety of source systems.
Data Asset : a collection of records within a Datasource which is usually named based on the underlying data system and sliced to correspond to a desired specification.

In [17]:
# 1. Load data cleaned result
raw_csv_path = "dataset_cleaned.csv"                    # Path for palm oil dataset
df_cleaned = pd.read_csv(raw_csv_path)          

# 2. Give a name to a Datasource. This name must be unique between Datasources
datasource_name = "csv-raw-data-palm-oil"
asset_name = "palm-oil-asset"
path_to_data = raw_csv_path
batch_name = "batch-definition-palm-oil"

# 3. Give a name to a data asset
datasource = context.data_sources.add_pandas(datasource_name)

# 4. Build batch definition
asset = datasource.add_dataframe_asset(asset_name)
batch_definition = asset.add_batch_definition_whole_dataframe(batch_name)

# 5. Read the csv file (same path as before) and prepare the batch parameters
df_cleaned = pd.read_csv(raw_csv_path)
batch_parameters = {"dataframe": df_cleaned}

## D.3 Create an Expectation Suite
An Expectation Suite is a collection of verifiable assertions about data. Expectation Suites combine multiple Expectations into an overall description of data.
Expectation Suite names are customizable, and the only constraint is that it must be unique to a given project.
You'll use a Validator to interact with your batch of data and generate an Expectation Suite.
Every time you evaluate an Expectation with validator.expect_*, it is immediately validated against your data.

In [18]:
# Create an expectation suite
expectation_suite_name = "expectation-palm-oil-dataset"
suite = context.suites.add(gx.ExpectationSuite(name=expectation_suite_name))

# Get a batch of data using the batch definition above
batch = batch_definition.get_batch(batch_parameters=batch_parameters)

# We will collect all expectations in this list, then save them into the suite
expectations = []

## D.4 Expectations
An Expectation is a verifiable assertion about source data. Similar to assertions in traditional Python unit tests, Expectations provide a flexible, declarative language for describing expected behaviors.Expectations is basically what do we expect from the data.
For example we expect our columns:
1. to be unique
2. to be between min_value and max_value
3. to be in set
4. to be in type list
5. to exist
6. most common value to be in set
7. value lengt to be between

In [19]:
# Add new column to achieve 'to be unique' requirement
df_cleaned["transaction_route_id"] = (
    df_cleaned["year"].astype(str) + "_" + 
    df_cleaned["mill_trase_id"].astype(str) + "_" +
    df_cleaned.index.astype(str))

In [20]:
# Expectation 1 : must be unique
expectation_1 = gxe.ExpectColumnValuesToBeUnique(column="transaction_route_id")
expectations.append(expectation_1)
batch.validate(expectation_1)

Calculating Metrics: 100%|██████████| 10/10 [00:01<00:00,  8.62it/s] 


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_unique",
    "kwargs": {
      "batch_id": "csv-raw-data-palm-oil-palm-oil-asset",
      "column": "transaction_route_id"
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 570434,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0,
    "partial_unexpected_counts": [],
    "partial_unexpected_index_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [21]:
# Expectation 2 : to be between min_value and max_value
expectation_2 = gxe.ExpectColumnValuesToBeBetween(column="volume", min_value=0, max_value=5000000)
expectations.append(expectation_2)
batch.validate(expectation_2)

Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 14.40it/s] 


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_between",
    "kwargs": {
      "batch_id": "csv-raw-data-palm-oil-palm-oil-asset",
      "column": "volume",
      "min_value": 0.0,
      "max_value": 5000000.0
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 570434,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0,
    "partial_unexpected_counts": [],
    "partial_unexpected_index_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [22]:
# Expectation 3 : to be in set

# 3.1 Check unique value in 'product_type' column
print(df_cleaned["product_type"].unique())

# 3.2 Running GX
expectation_3 = gxe.ExpectColumnValuesToBeInSet(column="product_type", value_set=["CRUDE PALM OIL", "REFINED PALM OIL"])
expectations.append(expectation_3)
batch.validate(expectation_3)

['CRUDE PALM OIL' 'REFINED PALM OIL']


Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 12.44it/s] 


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_in_set",
    "kwargs": {
      "batch_id": "csv-raw-data-palm-oil-palm-oil-asset",
      "column": "product_type",
      "value_set": [
        "CRUDE PALM OIL",
        "REFINED PALM OIL"
      ]
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 570434,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0,
    "partial_unexpected_counts": [],
    "partial_unexpected_index_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [23]:
# Expectation 4 : to be in type list
expectation_4 = gxe.ExpectColumnValuesToBeInTypeList(column="total_emission_exposure", type_list=["float", "float64", "integer"])
expectations.append(expectation_4)
batch.validate(expectation_4)

Calculating Metrics: 100%|██████████| 1/1 [00:00<00:00, 181.27it/s]


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_in_type_list",
    "kwargs": {
      "batch_id": "csv-raw-data-palm-oil-palm-oil-asset",
      "column": "total_emission_exposure",
      "type_list": [
        "float",
        "float64",
        "integer"
      ]
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "observed_value": "float64"
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [24]:
# Expectation 5 : column to exist
expectation_5 = gxe.ExpectColumnToExist(column="volume")
expectations.append(expectation_5)
batch.validate(expectation_5)

Calculating Metrics: 100%|██████████| 2/2 [00:00<00:00, 160.28it/s]


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_to_exist",
    "kwargs": {
      "batch_id": "csv-raw-data-palm-oil-palm-oil-asset",
      "column": "volume"
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {},
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [25]:
# Expectation 6 : most common value to be in set
expectation_6 = gxe.ExpectColumnMostCommonValueToBeInSet(column="province_of_production", value_set=["KALIMANTAN TENGAH"])
expectations.append(expectation_6)
batch.validate(expectation_6)

Calculating Metrics: 100%|██████████| 4/4 [00:00<00:00, 49.12it/s]  


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_most_common_value_to_be_in_set",
    "kwargs": {
      "batch_id": "csv-raw-data-palm-oil-palm-oil-asset",
      "column": "province_of_production",
      "value_set": [
        "KALIMANTAN TENGAH"
      ]
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "observed_value": [
      "KALIMANTAN TENGAH"
    ]
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [26]:
# Expectation 7 : value length to be between
expectation_7 = gxe.ExpectColumnValueLengthsToBeBetween(
    column="province_of_production_trase_id",
    min_value=4,
    max_value=10
)
expectations.append(expectation_7)
batch.validate(expectation_7)

Calculating Metrics: 100%|██████████| 11/11 [00:01<00:00,  8.30it/s] 


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_value_lengths_to_be_between",
    "kwargs": {
      "batch_id": "csv-raw-data-palm-oil-palm-oil-asset",
      "column": "province_of_production_trase_id",
      "min_value": 4,
      "max_value": 10
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 570434,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0,
    "partial_unexpected_counts": [],
    "partial_unexpected_index_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

All of the great expectations was succussfully 'true'

## D.5 Save Suite & Run Checkpoint
An Expectation is a verifiable assertion about source data. Similar to assertions in traditional Python unit tests, Expectations provide a flexible, declarative language for describing expected behaviors. Expectations is basically what do we expect from the data.
For example we expect our columns:
1. to be unique
2. to be between min_value and max_value
3. to be in set
4. to be in type list
5. to exist
6. most common value to be in set
7. value length to be between

In [27]:
# 1. Save all the expectations into Suite
for exp in expectations:
    suite.add_expectation(exp)
suite.save()

In [28]:
# 2. Create Validation Definition
name_validation = "validation-definition-palm-oil"
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(name=name_validation, data=batch_definition,suite=suite))

In [29]:
# 3. Create and Execute Checkpoint
checkpoint = context.checkpoints.add(
    gx.Checkpoint(name="palm_oil_checkpoint", validation_definitions=[validation_definition], actions=[gx.checkpoint.actions.UpdateDataDocsAction(name="update_data_docs")]))

In [30]:
checkpoint_result = checkpoint.run(batch_parameters=batch_parameters)
print(checkpoint_result.describe())

Calculating Metrics: 100%|██████████| 33/33 [00:03<00:00,  8.85it/s] 


{
    "success": true,
    "statistics": {
        "evaluated_validations": 1,
        "success_percent": 100.0,
        "successful_validations": 1,
        "unsuccessful_validations": 0
    },
    "validation_results": [
        {
            "success": true,
            "statistics": {
                "evaluated_expectations": 7,
                "successful_expectations": 7,
                "unsuccessful_expectations": 0,
                "success_percent": 100.0
            },
            "expectations": [
                {
                    "expectation_type": "expect_column_values_to_be_unique",
                    "success": true,
                    "kwargs": {
                        "batch_id": "csv-raw-data-palm-oil-palm-oil-asset",
                        "column": "transaction_route_id"
                    },
                    "result": {
                        "element_count": 570434,
                        "unexpected_count": 0,
                        "unexpected_p

In [31]:
spark.stop()